# 02 — Coleta de internações hospitalares (DATASUS SIH/SUS)

**Objetivo desta etapa:** obter, para os municípios de SP, o número de
internações de idosos (60+) por capítulos da CID-10 associados à hipótese do
estudo, entre 2022 e 2026.

**Status: já temos dado real.** Você exportou pelo TabNet 3 arquivos, um por
capítulo da CID-10 (é o nível de filtro que o TabNet oferece — não dá pra
pedir só W00-W19 ou só S72 direto na interface):

| Arquivo | Capítulo CID-10 | O que inclui |
|---|---|---|
| `sih_lesoes_sp.csv` | XIX — Lesões e causas externas | Quedas (W00-W19), fratura de fêmur (S72), e outras lesões/intoxicações |
| `sih_sintomas_sp.csv` | XVIII — Sintomas e sinais mal definidos | Síncope (R55), confusão mental (R41), e outros sintomas mal definidos |
| `sih_transtornos_mentais_sp.csv` | V — Transtornos mentais e comportamentais | Delirium (F05), e também outros transtornos mentais sem relação com isolamento |

⚠️ **Limitação a declarar no artigo:** cada capítulo é mais largo do que o
subgrupo de interesse original (quedas, fratura de fêmur, síncope, confusão
mental). O capítulo XVIII em especial é usado na literatura de saúde
pública como proxy de diagnóstico tardio/impreciso — o que reforça a
hipótese em vez de enfraquecê-la. Já o capítulo V é o mais largo dos três
(inclui transtornos por uso de substâncias, esquizofrenia etc.) — vale
tratar como complementar/exploratório, não como pilar central do
argumento. Ver `data/external/FONTES_RIO_CLARO.md`.

⚠️ **Mudança no recorte temporal:** o plano original era 2019-2022; o dado
real que conseguimos cobre 2022-2026 (2026 parcial, até julho, dados
provisórios). `config.ANOS_SIH` já reflete isso.

**Saída desta etapa:** `data/processed/internacoes_sp.csv`, com uma linha
por (município, ano, causa) e a contagem de internações.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import io


## 2.1 Parser do formato TabNet (testado com os arquivos reais)

O TabNet exporta em **Latin-1** (não UTF-8), separado por `;`, com algumas
linhas de cabeçalho/metadados antes da tabela e notas de rodapé depois —
por isso não dá pra usar `pd.read_csv` direto no arquivo, é preciso recortar
só o miolo da tabela primeiro. A função abaixo já foi validada linha a linha
contra os totais impressos no rodapé de cada CSV (batem exatamente).


In [ ]:
def parse_tabnet_sih(path, causa_label):
    """Lê um export do TabNet SIH (uma causa/capítulo, todos os municípios de SP)
    e devolve um DataFrame longo: codigo_datasus, ano, causa, internacoes."""
    with open(path, encoding="latin1") as f:
        lines = f.readlines()

    # a tabela começa na linha que tem o cabeçalho "Município" e termina
    # antes da linha "Total" (soma geral) + notas de rodapé
    header_idx = next(i for i, l in enumerate(lines) if l.startswith('"Munic'))
    end_idx = next(i for i, l in enumerate(lines) if l.startswith('"Total"'))
    chunk = "".join(lines[header_idx:end_idx])

    df = pd.read_csv(io.StringIO(chunk), sep=";", quotechar='"')
    df = df.rename(columns={df.columns[0]: "municipio_raw"})
    # a primeira coluna vem como "350010 ADAMANTINA" -- extrai só o código (6 dígitos)
    df["codigo_datasus"] = df["municipio_raw"].str.extract(r"^(\d{6})")

    anos = [c for c in df.columns if c.isdigit()]
    for col in anos:
        # "-" no TabNet significa zero absoluto
        df[col] = df[col].replace("-", "0").astype(int)

    longo = df.melt(id_vars=["codigo_datasus"], value_vars=anos, var_name="ano", value_name="internacoes")
    longo["ano"] = longo["ano"].astype(int)
    longo["causa"] = causa_label
    return longo[["codigo_datasus", "ano", "causa", "internacoes"]]


In [ ]:
partes = [
    parse_tabnet_sih(config.DATA_EXTERNAL / "sih_lesoes_sp.csv", "lesoes_causas_externas"),
    parse_tabnet_sih(config.DATA_EXTERNAL / "sih_sintomas_sp.csv", "sintomas_sinais_maldefinidos"),
    parse_tabnet_sih(config.DATA_EXTERNAL / "sih_transtornos_mentais_sp.csv", "transtornos_mentais"),
]
internacoes = pd.concat(partes, ignore_index=True)

print(internacoes.shape)
print(f"{internacoes['codigo_datasus'].nunique()} municípios distintos com alguma internação registrada")
print(internacoes.groupby("causa")["internacoes"].sum())
internacoes.head()


**Por que menos de 645 municípios aparecem:** o TabNet só lista uma linha para
o município quando ele teve pelo menos 1 internação naquela causa, no
período inteiro — municípios pequenos com 0 internações simplesmente não
aparecem no arquivo. Isso é tratado no notebook 03 (o merge parte da lista
completa de municípios e preenche com 0, em vez de partir daqui).


## 2.2 Salvar resultado consolidado


In [ ]:
internacoes.to_csv(config.DATA_PROCESSED / "internacoes_sp.csv", index=False)
print("Salvo em", config.DATA_PROCESSED / "internacoes_sp.csv")


## 2.3 (Opcional / mais adiante) Microdados via `pysus`

Se um dia quisermos refinar para o nível de subcategoria (só W00-W19, só
S72 etc., em vez do capítulo inteiro), o caminho é baixar os microdados de
AIH direto do SIH e classificar `DIAG_PRINC` nós mesmas — `config.py` já
tem `CAUSAS_CID10` e `classificar_causa()` prontos para isso. **Não é
necessário agora** (a seção 2.1 já resolve a coleta principal) — deixamos
aqui só como próximo passo possível para aumentar a precisão depois.


In [ ]:
try:
    from pysus.online_data.SIH import download as sih_download
    PYSUS_OK = True
except Exception as e:
    print("pysus não disponível (ok, não é necessário para o caminho principal):", e)
    PYSUS_OK = False

# Exemplo, se quiser tentar mais tarde:
# df_mes = sih_download(config.UF_SIGLA, 2022, 1)
# df_mes["causa"] = df_mes["DIAG_PRINC"].apply(config.classificar_causa)
